# Tarea 1 – Parte 2: Del LLM al clasificador

**Pregunta guía:** ¿se pueden convertir titulares de un diario peruano en un clasificador de noticias?

Este notebook parte de `datos/noticias.csv` (Parte 1: 376 noticias de El Comercio). La etiqueta de referencia (`etiqueta`) es la sección del diario mapeada a las 4 categorías oficiales. Sobre ella se comparan tres enfoques:

| Paso | Enfoque | Modelo |
| :--- | :--- | :--- |
| 1 | Línea base zero-shot (sin entrenamiento) | `Recognai/bert-base-spanish-wwm-cased-xnli` |
| 2 | Supervisión débil con LLM local | `Qwen2.5-1.5B-Instruct` (GGUF 4 bits, `llama.cpp`) |
| 3–5 | Transformer con fine-tuning | `distilbert/distilbert-base-multilingual-cased` |
| 6 | Comparativa y decisión | — |

**Equipo usado:** laptop con Windows, **solo CPU** (AMD Ryzen 5 5500U, 6 núcleos) y **7.3 GB de RAM**. Esta restricción explica varias decisiones técnicas, que se documentan en cada sección: el formato del LLM, el modelo zero-shot en español en lugar de `xlm-roberta-large` (2.2 GB) y los embeddings congelados en el fine-tuning.

In [1]:
import gc
import json
import shutil
import time
import warnings
from datetime import date
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")

import truststore

# Valida HTTPS con los certificados del sistema (el antivirus intercepta HTTPS con su propio
# certificado raíz); lo necesita huggingface_hub para consultar los modelos.
truststore.inject_into_ssl()

import datasets
import huggingface_hub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns
import transformers
from datasets import Dataset
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, pipeline, set_seed)

# Salida limpia: sin barras de progreso de carga y guardado. También se silencia un aviso de
# transformers 5 sobre un "regex incorrecto" en el tokenizador, que es un falso positivo para
# DistilBERT (usa BertPreTokenizer + WordPiece).
transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
datasets.disable_progress_bars()
huggingface_hub.utils.disable_progress_bars()
huggingface_hub.utils.logging.set_verbosity_error()  # sin el aviso de "requests sin HF_TOKEN"

pio.renderers.default = "plotly_mimetype+png"
pio.renderers["png"].width = 950
pio.templates.default = "plotly_white"
pd.set_option("display.max_colwidth", 110)

SEMILLA = 42
set_seed(SEMILLA)
DATOS, RAW, MODELO = Path("datos"), Path("datos/raw"), Path("modelo")
ETIQUETAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]

df = pd.read_csv(DATOS / "noticias.csv").fillna({"bajada": ""})
print("Fecha de ejecución:", date.today().isoformat(), "| df.shape:", df.shape)
df["etiqueta"].value_counts().reindex(ETIQUETAS).rename("noticias")

Fecha de ejecución: 2026-10-18 | df.shape: (376, 7)


etiqueta
politica              92
economia              90
deportes              98
ciencia_tecnologia    96
Name: noticias, dtype: int64

In [2]:
def metricas(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, labels=ETIQUETAS, average="macro", zero_division=0),
    }

## 1. Línea base zero-shot (sin fine-tuning)

Se evalúan **40 titulares** (10 por categoría, muestreo estratificado con `random_state=42`) con `pipeline("zero-shot-classification")` y el modelo NLI en español `Recognai/bert-base-spanish-wwm-cased-xnli`. Se eligió esta opción del enunciado en lugar de `joeddav/xlm-roberta-large-xnli`, que pesa 2.2 GB y no cabe con holgura en la RAM disponible.

El modelo no conoce las categorías. Para cada titular evalúa si se cumple la hipótesis *"Esta noticia trata sobre {categoría}."*. Se usan los nombres naturales en español (`política`, `economía`, `deportes`, `ciencia y tecnología`) y luego se mapean a las etiquetas oficiales. El F1 macro que se obtiene es el **piso mínimo** que deben superar los otros enfoques.

In [3]:
ZS_MODELO = "Recognai/bert-base-spanish-wwm-cased-xnli"
ZS_ETIQUETAS = {"política": "politica", "economía": "economia",
                "deportes": "deportes", "ciencia y tecnología": "ciencia_tecnologia"}
zs = pipeline("zero-shot-classification", model=ZS_MODELO, device=-1)


def zero_shot(titulares):
    salida = zs(list(titulares), candidate_labels=list(ZS_ETIQUETAS),
                hypothesis_template="Esta noticia trata sobre {}.")
    return [ZS_ETIQUETAS[s["labels"][0]] for s in salida]


muestra_zs = df.groupby("etiqueta", group_keys=False).sample(n=10, random_state=SEMILLA).copy()
t0 = time.time()
muestra_zs["pred_zs"] = zero_shot(muestra_zs["titular"])
lat_zs40 = (time.time() - t0) / len(muestra_zs)
m_zs40 = metricas(muestra_zs["etiqueta"], muestra_zs["pred_zs"])
print(f"Zero-shot en 40 titulares -> accuracy {m_zs40['accuracy']:.3f} | F1 macro {m_zs40['f1_macro']:.3f} "
      f"| {lat_zs40 * 1000:.0f} ms por titular")
pd.crosstab(muestra_zs["etiqueta"], muestra_zs["pred_zs"], rownames=["real"], colnames=["zero-shot"])

Zero-shot en 40 titulares -> accuracy 0.650 | F1 macro 0.634 | 351 ms por titular


zero-shot,ciencia_tecnologia,deportes,economia,politica
real,,,,
ciencia_tecnologia,6,2,1,1
deportes,0,8,0,2
economia,1,1,3,5
politica,0,0,1,9


In [4]:
# Se libera memoria: el modelo zero-shot se vuelve a cargar en la comparativa final
del zs
gc.collect();